# Assignment 2B — RAG Pipeline (Manufacturing & Technical Standards)

**Course:** Large Language Models for Generative AI (S1-26_AIMLZG536)  **Group:** 78

Builds a small, complete Retrieval-Augmented Generation (RAG) system on the Assignment 1 domain corpus:

```
12 cleaned .txt docs ──► chunk ──► embed ──► FAISS (dense) ─┐
                              └──► BM25 (keyword) ──────────┼─► RRF hybrid ─► cross-encoder rerank ─► top-3 chunks
                                                                                                         │
                         Assignment 1 model (TinyLlama-1.1B + QLoRA Adapter B) ◄── grounded prompt ◄──────┘
                                         │
                                         └──► LLM judge (correct? supported?)
```

| Part | What we do | Marks |
|---|---|---|
| A | Dense search + choose a chunk size | 5 |
| B | Dense vs BM25 vs Hybrid (RRF) | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded generation + LLM judge | 7 |

**Reused from Assignment 1:** the 12 cleaned domain `.txt` files, the 10-pair evaluation set (`eval_dataset.jsonl`, seed-42 split), and the fine-tuned TinyLlama-1.1B + QLoRA Adapter B.

---
## Step 0 — Environment setup
Parts A–C run on CPU or GPU. Part D loads LLMs and needs a GPU (Prayogshala / Colab).

In [ ]:
%pip install -q sentence-transformers faiss-cpu rank_bm25

In [ ]:
import os, re, json, time, random, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import torch

random.seed(42); np.random.seed(42); torch.manual_seed(42)
pd.set_option("display.max_colwidth", 120)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only")

# Find the corpus folder whichever platform we are on
CANDIDATES = [Path("domain_corpus"), Path("/home/jovyan/domain_corpus"), Path("/content/domain_corpus")]
CORPUS_DIR = next((p for p in CANDIDATES if p.exists() and any(p.glob("*.txt"))), None)
assert CORPUS_DIR is not None, "Upload the 12 cleaned .txt files into a folder named domain_corpus/"
print("Corpus folder:", CORPUS_DIR.resolve())

### 0.1 Load the document collection
We reuse the cleaned Assignment 1 files as-is. Only whitespace is collapsed — we deliberately do **not** fix the OCR noise
(e.g. `management-ied`, `weii`) because real RAG systems must cope with imperfect text, and the answer spans below are
copied verbatim from this text.

In [ ]:
def normalize_ws(text: str) -> str:
    # Collapse all runs of whitespace (newlines, tabs, double spaces) to a single space.
    return re.sub(r"\s+", " ", text).strip()

docs = {p.name: normalize_ws(p.read_text(encoding="utf-8", errors="ignore"))
        for p in sorted(CORPUS_DIR.glob("*.txt"))}

corpus_stats = pd.DataFrame([{"document": n, "characters": len(t), "words": len(t.split())} for n, t in docs.items()])
print(f"{len(docs)} documents | {corpus_stats.characters.sum():,} characters | {corpus_stats.words.sum():,} words")
corpus_stats

### 0.2 The 10 evaluation queries and their answer spans
The queries and expected answers are the 10-pair evaluation split from Assignment 1 (48 curated pairs, 80/20 split, `random.seed(42)`).

**Answer span** = the sentence or short passage in the corpus that answers the query. A retrieved chunk is a **hit** if it contains the span.
Each span was found by searching the query's source document and copied verbatim (OCR errors included). Spans are kept short
(13–27 tokens) so that each one can fit inside even a 100-token chunk.

In [ ]:
EVAL = [
 {"qid": 1, "source": "NIST_Malcolm_Baldrige_Quality.txt",
  "query": "How did Malcolm Baldrige quality concepts influence the STEP model qualification approach?",
  "expected": "The STEP qualification effort used several ideas drawn from the Malcolm Baldrige National Quality Award criteria. These ideas encouraged measurable quality thresholds, links between processes and results, continuous improvement, comparison with established performance levels, and management involvement in producing high-quality information models.",
  "span": "Many basic concepts of the Malcolm Baldrige National Quality Award were used in building the foundation for this new approach to standards development"},
 {"qid": 2, "source": "OSHA_Ergonomics_Workplace.txt",
  "query": "How can workstation design reduce excessive reaching and awkward upper-body posture?",
  "expected": "Workstations can be arranged so materials and products are positioned closer to workers. Examples include conveyor diverters, appropriately located chutes or cut-outs, and correctly adjusted work platforms. These changes can reduce long reaches, elevated arms, wrist deviation, forward bending, and other stressful postures.",
  "span": "This conveyor has diverters in place that push the products closer to the workers. This reduces reaches for the employees."},
 {"qid": 3, "source": "NIST_Malcolm_Baldrige_Quality.txt",
  "query": "What three kinds of evidence can be used to assess results under the Baldrige-inspired approach described in the report?",
  "expected": "The report describes results as being assessed through improvement trends, current performance levels, and benchmarks or other comparisons with external performance. Together these forms of evidence show whether quality is improving, what level has been reached, and how performance compares with appropriate leaders.",
  "span": "based upon one or more of three factors: (1) improvement trends; (2) current levels; and (3) benchmarks"},
 {"qid": 4, "source": "NIST_Manufacturing_Handbook.txt",
  "query": "Why are statistical methods useful when analysing manufacturing experiments?",
  "expected": "Statistical methods provide a systematic way to summarize experimental observations, describe the amount of variation in measured results, and judge whether apparent differences are meaningful. This helps manufacturing engineers draw conclusions from data rather than relying only on individual observations or subjective impressions.",
  "span": "One of the most frequent uses of statistics is in testing for differences."},
 {"qid": 5, "source": "NIST_Manufacturing_Machinery_Maintenance.txt",
  "query": "What is the difference between reactive maintenance and more advanced maintenance strategies?",
  "expected": "Reactive maintenance responds after machinery has already failed or developed a problem. Preventive and predictive approaches attempt to reduce failures by maintaining equipment before breakdowns occur or by using information about equipment condition to guide maintenance decisions, thereby reducing avoidable downtime and production losses.",
  "span": "is maintenance done, typically, after equipment has failed to produce a product within desired quality or production targets"},
 {"qid": 6, "source": "NIST_Manufacturing_Machinery_Maintenance.txt",
  "query": "What costs are included when analysing direct and indirect machinery-maintenance impacts?",
  "expected": "Direct maintenance costs include labour and materials used for maintenance and repair. The broader economic analysis also considers cascading damage following breakdowns, inventory increases caused by uncertainty or disruption, unplanned downtime, defects, lost sales, and other losses associated with faults and failures.",
  "span": "Direct maintenance and repair costs include the cost of labor and materials along with cascading effects"},
 {"qid": 7, "source": "NIST_Measurement_Uncertainty.txt",
  "query": "What is the difference between Type A and Type B evaluations of uncertainty?",
  "expected": "A Type A evaluation determines an uncertainty component through statistical analysis of repeated observations. A Type B evaluation uses other available information, such as calibration data, previous measurements, instrument specifications, reference data, or scientific judgement, rather than relying on repeated statistical observations alone.",
  "span": "The evaluation of uncertainty by the statistical analysis of series of observations is termed a Type A evaluation"},
 {"qid": 8, "source": "NIST_ISO9000_Quality_Systems.txt",
  "query": "What common elements are included in the ISO 9001, ISO 9002, and ISO 9003 quality-system models?",
  "expected": "The ISO 9000 quality-system models share requirements for an effective quality system, valid measurements, regularly calibrated measuring and testing equipment, suitable statistical techniques, product identification and traceability, adequate records, inspection and testing, proper handling and storage, and control of nonconforming items.",
  "span": "Common elements in ISO 9001, 9002, and 9003 include the need for; an effective quality system"},
 {"qid": 9, "source": "NIST_Malcolm_Baldrige_Quality.txt",
  "query": "Why is management involvement important in a quality-improvement program?",
  "expected": "The report emphasizes that successful quality improvement should be management-led and customer-oriented. Management establishes expectations, provides direction and resources, supports the use of quality metrics and thresholds, and creates the organizational conditions needed for persistent, systematic improvement rather than isolated corrective actions.",
  "span": "quality improvement programs must be management-ied and customer-oriented"},
 {"qid": 10, "source": "OSHA_Machine_Guarding_Standards.txt",
  "query": "What is the main purpose of machine guarding?",
  "expected": "Machine guarding protects operators and other employees from hazards created by moving machine parts and operations. Safeguards are intended to prevent workers from reaching dangerous areas such as points of operation, rotating components, nip points, and other locations where cutting, crushing, or entanglement injuries could occur.",
  "span": "Safeguarding is essential for protecting employees from needless and preventable injury."},
]
QUERIES = [e["query"] for e in EVAL]

# Hit test used in Parts A, B and C: case- and whitespace-insensitive substring match
def norm(s: str) -> str:
    return normalize_ws(s).lower()

def is_hit(chunk_text: str, span: str) -> bool:
    return norm(span) in norm(chunk_text)

# Sanity check: every span must exist (exactly once) in its source document
for e in EVAL:
    n = norm(docs[e["source"]]).count(norm(e["span"]))
    assert n >= 1, f"Q{e['qid']}: span not found in {e['source']}"
    e["span_occurrences"] = n

pd.DataFrame(EVAL)[["qid", "query", "source", "span", "span_occurrences"]]

---
# Part A — Dense search and chunk size (5 marks)

**Dense retrieval in one paragraph.** An embedding model maps any text to a fixed-length vector so that texts with similar
*meaning* land close together. We embed every chunk once (offline), store the vectors in an index, and at query time embed the
query and return the chunks whose vectors are most similar. If all vectors are L2-normalised, the inner product equals cosine
similarity, so a FAISS `IndexFlatIP` (exact, brute-force inner-product search) gives exact cosine-similarity search.

### Embedding model choice and its input limit
The assignment asks for chunk sizes of 100 and 400 tokens. `all-MiniLM-L6-v2` truncates input at **256** tokens, so a
400-token chunk would lose its last ~144 tokens when embedded (they would still be sent to the LLM, but would be invisible to search).
**We therefore switched to a model with a longer limit: `BAAI/bge-small-en-v1.5`** — 512-token limit, 384-dim vectors,
33M parameters, so it is just as cheap to run. Tokens are counted with this model's own tokenizer.

BGE models are trained with an instruction prefix for short queries
(`"Represent this sentence for searching relevant passages: "`); passages are embedded without it.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX   = "Represent this sentence for searching relevant passages: "
EMBED_BATCH    = 64

embedder  = SentenceTransformer(EMBED_MODEL_ID, device=DEVICE)
tokenizer = embedder.tokenizer
MAX_TOKENS = embedder.max_seq_length            # limit applied when embedding (incl. [CLS]/[SEP])
EMBED_DIM  = embedder.get_sentence_embedding_dimension()

print("Embedding model :", EMBED_MODEL_ID)
print("Max input length:", MAX_TOKENS, "tokens")
print("Vector size     :", EMBED_DIM)

def count_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

## A1. Build dense search (2 marks)

### A1.1 — One reusable dense-search component
`DenseIndex` embeds a list of chunks, builds a FAISS `IndexFlatIP`, and `search(query, k)` returns the top-k chunks with scores.
The same class is reused for every chunk size in A2 and again in Parts B and C.

In [ ]:
class DenseIndex:
    # Embed chunks -> FAISS IndexFlatIP (cosine via normalised vectors) -> top-k search.

    def __init__(self, chunks, model=embedder, batch_size=EMBED_BATCH, query_prefix=QUERY_PREFIX):
        self.chunks, self.model, self.batch_size, self.query_prefix = chunks, model, batch_size, query_prefix
        texts = [c["text"] for c in chunks]

        t0 = time.perf_counter()
        vecs = model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=False).astype("float32")
        t1 = time.perf_counter()
        self.index = faiss.IndexFlatIP(vecs.shape[1])
        self.index.add(vecs)
        t2 = time.perf_counter()

        self.embed_seconds, self.add_seconds = t1 - t0, t2 - t1
        self.build_seconds = t2 - t0

    def search(self, query: str, k: int = 5):
        q = self.model.encode([self.query_prefix + query], normalize_embeddings=True,
                              convert_to_numpy=True).astype("float32")
        scores, ids = self.index.search(q, k)
        return [{**self.chunks[i], "score": float(s), "rank": r}
                for r, (i, s) in enumerate(zip(ids[0], scores[0]), start=1)]

## A2. Choose a chunk size (3 marks)

### Fixed-size chunking with the embedding model's tokenizer
Each document is tokenized once with `return_offsets_mapping=True`. That gives, for every token, its character position in the
original text, so a window of N tokens can be cut directly from the **original** text (no decode step, so casing and
punctuation are preserved). Chunks never cross document boundaries, and there is no overlap, as the assignment specifies.

Both sizes fit the model: 100 + 2 and 400 + 2 special tokens are ≤ 512, so nothing is truncated.

In [ ]:
def chunk_fixed(docs: dict, size: int):
    # Split every document into consecutive, non-overlapping windows of `size` tokens.
    chunks = []
    for name, text in docs.items():
        enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True, verbose=False)
        offsets = enc["offset_mapping"]
        for start in range(0, len(offsets), size):
            window = offsets[start:start + size]
            chunks.append({
                "chunk_id": len(chunks),
                "doc": name,
                "text": text[window[0][0]:window[-1][1]],
                "n_tokens": len(window),
            })
    return chunks

CHUNK_SIZES = [100, 400]
chunk_sets = {size: chunk_fixed(docs, size) for size in CHUNK_SIZES}

for size, chunks in chunk_sets.items():
    assert max(c["n_tokens"] for c in chunks) + 2 <= MAX_TOKENS
    print(f"{size:>3}-token chunks: {len(chunks):>5} chunks | example from {chunks[1]['doc']}:")
    print(textwrap.indent(textwrap.fill(chunks[1]["text"][:300] + " ...", 110), "    "))

**Can every span be found at each size?** With no overlap, a span that happens to straddle a chunk boundary is split in two,
and *no* chunk can ever contain it. Checking this up front tells us the best hit rate each chunk size could possibly reach.

In [ ]:
reachable = pd.DataFrame({
    f"{size} tokens": ["yes" if any(is_hit(c["text"], e["span"]) for c in chunks) else "SPLIT"
                       for e in EVAL]
    for size, chunks in chunk_sets.items()
}, index=[f"Q{e['qid']}" for e in EVAL])
print("Span contained intact in at least one chunk:")
reachable.T

### A2.1 — Dense search at 100 vs 400 tokens (2 marks)
For each size: build the dense index (A1), retrieve the top 3 for each of the 10 queries, and compute
* **Hit rate@3** — share of queries where one of the top-3 chunks contains the answer span
* **Tokens sent per query** — combined token count of the top-3 chunks, averaged over the 10 queries

In [ ]:
def hit_at_k(results, span, k):
    return any(is_hit(r["text"], span) for r in results[:k])

dense_indexes, a2_rows, a2_detail = {}, [], []
for size, chunks in chunk_sets.items():
    idx = DenseIndex(chunks)
    dense_indexes[size] = idx
    hits, tokens_sent = [], []
    for e in EVAL:
        top3 = idx.search(e["query"], k=3)
        hit_rank = next((r["rank"] for r in top3 if is_hit(r["text"], e["span"])), None)
        hits.append(hit_rank is not None)
        tokens_sent.append(sum(r["n_tokens"] for r in top3))
        a2_detail.append({"chunk size": size, "qid": e["qid"], "hit rank (top 3)": hit_rank or "-",
                          "top-1 doc": top3[0]["doc"], "correct doc": e["source"]})
    a2_rows.append({"Chunk size (tokens)": size,
                    "Total chunks": len(chunks),
                    "Hit rate@3 (%)": 100 * np.mean(hits),
                    "Tokens sent per query (top 3)": round(np.mean(tokens_sent), 1),
                    "Index build time (s)": round(idx.build_seconds, 2)})

a2_table = pd.DataFrame(a2_rows)
a2_table

In [ ]:
# Per-query view: which queries hit, and at what rank? Did the top-1 chunk even come from the right document?
pd.DataFrame(a2_detail).pivot(index="qid", columns="chunk size", values=["hit rank (top 3)", "top-1 doc"])

### A2.2 — Chosen chunk size (1 mark)
> ✍️ **TODO (write after running):** two or three sentences naming the chunk size used in Parts B–D and why.
> Weigh (1) hit rate@3, (2) tokens sent per query, and (3) the context budget of the Assignment 1 model —
> TinyLlama-1.1B has a **2,048-token** context window, which must hold the instructions, 3 chunks, the question
> *and* the generated answer (~200 tokens). Also look at the "SPLIT" table above: does one size lose spans at boundaries?

In [ ]:
CHOSEN_SIZE = 400   # <- set after reading the A2 table (100 or 400)

TINYLLAMA_CONTEXT = 2048
avg_sent = a2_table.set_index("Chunk size (tokens)").loc[CHOSEN_SIZE, "Tokens sent per query (top 3)"]
print(f"Chosen chunk size: {CHOSEN_SIZE} tokens")
print(f"Top-3 context ≈ {avg_sent:.0f} embedding-tokens ≈ {100 * avg_sent / TINYLLAMA_CONTEXT:.0f}% of TinyLlama's {TINYLLAMA_CONTEXT}-token window")

CHUNKS = chunk_sets[CHOSEN_SIZE]
dense  = dense_indexes[CHOSEN_SIZE]

### A1.2 — Dense search report for the final chunk set (1 mark)

In [ ]:
a1_report = pd.DataFrame([
    ("Embedding model", EMBED_MODEL_ID),
    ("Max input length (tokens)", MAX_TOKENS),
    ("Vector size", EMBED_DIM),
    ("Similarity measure", "Cosine similarity (L2-normalised vectors + inner product, FAISS IndexFlatIP, exact search)"),
    ("Batch size", EMBED_BATCH),
    ("Device", DEVICE),
    ("Final chunk set", f"{CHOSEN_SIZE}-token fixed chunks, no overlap, {len(CHUNKS):,} chunks"),
    ("Index build time (s)", f"{dense.build_seconds:.2f}  (embedding {dense.embed_seconds:.2f} + index add {dense.add_seconds:.4f})"),
], columns=["Item", "Value"])
a1_report

> **Observation:** almost all of the build time is spent *embedding* the chunks (a forward pass of the model per chunk);
> adding vectors to a flat FAISS index is just a memory copy. The flat index is exact; at this corpus size (a few thousand
> vectors) approximate indexes such as HNSW or IVF are unnecessary.